In [1]:
import os
import re
import pandas as pd

def extrair_dados_instancia(pasta_resultados):
    melhores_resultados = {}

    # Expressões regulares para capturar o Custo e os Bins
    re_bins = re.compile(r'Bins:\s*([\d\s,]+)')
    re_cost = re.compile(r'Cost:\s*(\d+)')

    # Percorre todos os arquivos contidos na pasta informada
    for nome_arquivo in os.listdir(pasta_resultados):
        # Verifica se o arquivo segue o padrão esperado: LNS_instance={instancia}_{rodada}
        match_nome = re.match(r'LNS_instance=(\d+)_(\d+)', nome_arquivo)
        if not match_nome:
            continue
        
        instancia = int(match_nome.group(1))
        rodada = int(match_nome.group(2))
        
        caminho_completo = os.path.join(pasta_resultados, nome_arquivo)
        
        try:
            with open(caminho_completo, 'r', encoding='utf-8') as f:
                conteudo = f.read()
                
                # Busca o custo e os bins no corpo do arquivo
                match_cost = re_cost.search(conteudo)
                match_bins = re_bins.search(conteudo)
                
                if match_cost and match_bins:
                    custo = int(match_cost.group(1))
                    
                    # Processa a string de bins para contar a quantidade de números
                    bins_str = match_bins.group(1).strip()
                    # Divide pelas vírgulas e remove elementos vazios causados por vírgula no final
                    lista_bins = [b.strip() for b in bins_str.split(',') if b.strip()]
                    num_bins = len(lista_bins)
                    
                    # Critério de seleção do melhor resultado para a instância:
                    # Menor Custo. Se empatar, escolhe o que tiver Menor número de Bins.
                    if instancia not in melhores_resultados:
                        melhores_resultados[instancia] = {
                            'Instancia': instancia,
                            'Melhor_Rodada': rodada,
                            'Custo': custo,
                            'Num_Bins': num_bins,
                            'Arquivo': nome_arquivo
                        }
                    else:
                        atual = melhores_resultados[instancia]
                        # Atualiza se encontrar um custo menor OU custo igual com menos Bins
                        if (custo < atual['Custo']) or (custo == atual['Custo'] and num_bins < atual['Num_Bins']):
                            melhores_resultados[instancia] = {
                                'Instancia': instancia,
                                'Melhor_Rodada': rodada,
                                'Custo': custo,
                                'Num_Bins': num_bins,
                                'Arquivo': nome_arquivo
                            }
        except Exception as e:
            print(f"Erro ao ler o arquivo {nome_arquivo}: {e}")

    # Ordena o dicionário pelas chaves das instâncias (1 a 25)
    resultados_finais = [melhores_resultados[i] for i in sorted(melhores_resultados.keys())]
    return resultados_finais

# Configuração da pasta de entrada
pasta = 'results_20M'

print("Iniciando o processamento dos arquivos...")
dados_extraidos = extrair_dados_instancia(pasta)

# Exibição e exportação dos resultados obtidos
if dados_extraidos:
    df = pd.DataFrame(dados_extraidos)
    
    print("\n" + "="*23 + " MELHORES RESULTADOS POR INSTÂNCIA " + "="*23)
    print(df[['Instancia', 'Melhor_Rodada', 'Custo', 'Num_Bins', 'Arquivo']].to_string(index=False))
    
    # Salva os resultados compilados em uma planilha CSV
    df.to_csv('melhores_resultados.csv', index=False, encoding='utf-8')
    print("\nResultados consolidados salvos com sucesso em 'melhores_resultados.csv'!")
else:
    print(f"\n[Aviso] Nenhum arquivo compatível foi encontrado na pasta '{pasta}'.")
    print("Verifique se o script está na mesma pasta onde 'results_20M' está localizada.")

Iniciando o processamento dos arquivos...

======================= MELHORES RESULTADOS POR INSTÂNCIA =======================
 Instancia  Melhor_Rodada  Custo  Num_Bins                Arquivo
         1              8   7746         4   LNS_instance=1_8.txt
         2              4   7581         4   LNS_instance=2_4.txt
         3              1   7465         4   LNS_instance=3_1.txt
         4              6  11957         5   LNS_instance=4_6.txt
         5             10  11543         4  LNS_instance=5_10.txt
         6              1  11430         4   LNS_instance=6_1.txt
         7              2   5666         4   LNS_instance=7_2.txt
         8              8   5367         4   LNS_instance=8_8.txt
         9              1   5367         4   LNS_instance=9_1.txt
        10              5   8302         5  LNS_instance=10_5.txt
        11              8   7864         4  LNS_instance=11_8.txt
        12              5   7697         4  LNS_instance=12_5.txt
        13       

In [2]:
def encontrar_arquivo_instancia(numero_instancia, pasta_instances='instances'):
    """
    Busca na pasta especificada o arquivo correspondente ao número da instância.
    O formato esperado do nome é: instance={numero_instancia}_{resto}.txt
    
    Retorna o caminho completo do arquivo se encontrado, ou None caso contrário.
    """
    # Cria o padrão de busca, ex: "instances/instance=1_*.txt"
    padrao_busca = os.path.join(pasta_instances, f"instance={numero_instancia}_*.txt")
    
    # Procura por todos os arquivos que casam com o padrão
    arquivos_encontrados = glob.glob(padrao_busca)
    
    if arquivos_encontrados:
        # Se houver mais de um (o que não deve acontecer), pega o primeiro
        return arquivos_encontrados[0]
    else:
        print(f"[Aviso] Arquivo para a instância {numero_instancia} não foi encontrado na pasta '{pasta_instances}'.")
        return None

In [ ]:
import glob
import subprocess
import random
pasta_instancias_originais = 'instances'
file = 'program'

for dados in dados_extraidos:
    num_instancia = dados['Instancia']
    melhor_custo = dados['Custo']
    num_bins = dados['Num_Bins']
    
    caminho_instancia = encontrar_arquivo_instancia(num_instancia, pasta_instancias_originais)

    # Sem o stdout=PIPE, ele joga direto no terminal
    proc = subprocess.Popen(
        [f"./{file}", caminho_instancia, str(melhor_custo), str(num_bins)]
    )
    
    # É uma boa prática esperar o processo terminar antes de ir para a próxima instância
    proc.wait()

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_UpperCutoff              7746
Tried aggregator 2 times.
MIP Presolve eliminated 4660 rows and 32 columns.
MIP Presolve modified 22 coefficients.
Aggregator did 325 substitutions.
Reduced MIP has 3192 rows, 408 columns, and 9966 nonzeros.
Reduced MIP has 408 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (20.91 ticks)
Probing time = 0.00 sec. (1.28 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 3192 rows, 408 columns, and 9966 nonzeros.
Reduced MIP has 408 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (5.44 ticks)
Probing time = 0.00 sec. (1.28 ticks)
Clique table members: 72.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 12 threads.
Root relaxation solution time = 0.03 sec. (38.79 ticks)

        N

Default variable names x1, x2 ... being created.
Default row names c1, c2 ... being created.


      0     0     4402.3235   232                   Cuts: 308      882         
      0     0     4438.3024   179                    Cuts: 90     1115         
      0     0     4440.1711   205                   Cuts: 199     1271         
      0     0     4441.2146   208                   Cuts: 106     1342         
      0     0     4441.2985   190                    Cuts: 77     1396         
Detecting symmetries...
      0     0     4459.0051   158                   Cuts: 163     1622         
      0     0     4460.0538   165                    Cuts: 92     1699         
      0     0     4460.0538   168                ZeroHalf: 85     1731         
      0     0     4460.0538   171                ZeroHalf: 99     1770         
Detecting symmetries...
      0     2     4460.0538   171                   4470.3836     1770         
Elapsed time = 1.06 sec. (890.99 ticks, tree = 0.02 MB, solutions = 0)
     28     8     5130.1361   153                   4544.3989     4625         
 

Default variable names x1, x2 ... being created.
Default row names c1, c2 ... being created.


Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_TimeLimit                               7200
CPXPARAM_MIP_Tolerances_UpperCutoff              7465
Tried aggregator 2 times.
MIP Presolve eliminated 4660 rows and 32 columns.
MIP Presolve modified 22 coefficients.
Aggregator did 325 substitutions.
Reduced MIP has 3192 rows, 568 columns, and 10606 nonzeros.
Reduced MIP has 568 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (22.64 ticks)
Probing time = 0.00 sec. (0.78 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 3192 rows, 568 columns, and 10606 nonzeros.
Reduced MIP has 568 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (6.10 ticks)
Probing time = 0.00 sec. (0.78 ticks)
Clique table members: 154.
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 12 threads.
Root relaxation solution time = 0.04 sec. (58.99 ticks)

      

Default variable names x1, x2 ... being created.
Default row names c1, c2 ... being created.


Root relaxation solution time = 0.08 sec. (110.46 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

      0     0     7264.8760   162                   7264.8760     1320         
      0     0     7322.2152   188                   Cuts: 406     1738         
      0     0     7350.4035   164                   Cuts: 175     2062         
      0     0     7351.0775   195                   Cuts: 159     2170         
      0     0     7351.6019   203                   Cuts: 102     2257         
      0     0     7351.6019   215                ZeroHalf: 91     2295         
Detecting symmetries...
      0     0     7351.6019   207                ZeroHalf: 61     2320         
      0     0     7372.6624   238                   Cuts: 147     2700         
      0     0     7373.4655   270                   Cuts: 154     2828         
      0     0     7401.7091   269                   Cuts: 